# Yahoo Fantasy Connection

1. Paste your Client ID and Client Secret into `oauth2.json` (project root).
2. Run the cells in order. The first time, a browser opens: approve access, copy the code Yahoo shows, and paste it into the input box that appears here.
3. After that the token is saved into `oauth2.json` and refreshed automatically.

In [1]:
import json
from pathlib import Path

import pandas as pd
import yahoo_fantasy_api as yfa
from yahoo_oauth import OAuth2

# Find oauth2.json whether the notebook runs from notebooks/ or the project root
CREDS = next(p / "oauth2.json" for p in [Path.cwd(), *Path.cwd().parents] if (p / "oauth2.json").exists())

creds = json.loads(CREDS.read_text())
if creds["consumer_key"].startswith("PASTE_"):
    raise ValueError(f"Fill in your Yahoo Client ID and Secret in {CREDS} first.")
print("Using credentials from", CREDS)

Using credentials from c:\Users\Jesli's Laptop\Desktop\coding-projects\fantasy-tools\oauth2.json


Then, after you approve access, the browser will go to a page that won't load. That's expected. Copy the value after code= in the address bar and paste that as the code.

In [2]:
sc = OAuth2(None, None, from_file=str(CREDS))
if not sc.token_is_valid():
    sc.refresh_access_token()
print("Connected to Yahoo")

[2026-09-23 15:48:24,415 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-09-23 15:48:24,418 DEBUG] [yahoo_oauth.oauth.handler] AUTHORIZATION URL : https://api.login.yahoo.com/oauth2/request_auth?redirect_uri=https%3A%2F%2Flocalhost%3A8080&response_type=code&client_id=dj0yJmk9dERJODdNNzV3djhQJmQ9WVdrOVlrZHdkbWw1UzFnbWNHbzlNQT09JnM9Y29uc3VtZXJzZWNyZXQmc3Y9MCZ4PWVj
[2026-09-23 15:49:07,116 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 5.317789554595947
[2026-09-23 15:49:07,116 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


Connected to Yahoo


In [3]:
gm = yfa.Game(sc, "nba")
league_ids = gm.league_ids()
print("Your NBA leagues (all seasons):", league_ids or "none yet - join a league first")

RuntimeError: b'{\n    "error": {\n        "xml:lang": "en-us",\n        "yahoo:uri": "\\/fantasy\\/v2\\/users\\/games\\/leagues?use_login=1&amp;is_available=0&amp;game_types=&amp;game_codes=&amp;seasons=&amp;format=json",\n        "description": "This application is not authorized to perform this action.",\n        "detail": ""\n    }\n}'

In [ ]:
# Most recent league; change the index if you're in more than one
lg = gm.to_league(league_ids[-1])
settings = lg.settings()
print(settings["name"], "|", settings.get("scoring_type"), "|", settings.get("num_teams"), "teams")

stat_cats = [c["display_name"] for c in lg.stat_categories()]
print("Categories:", stat_cats)

In [ ]:
# My roster (empty until after your draft)
my_team = lg.to_team(lg.team_key())
pd.DataFrame(my_team.roster())